这个代码2025年10月重启EPFLUX计算的测试代码，只使用ERA5数据进行计算看计算程序是否正确。

In [ ]:
# -*- coding: utf-8 -*-
import os, time, cdsapi, xarray as xr, itertools, glob
from datetime import date

# ================= 基本配置（按需修改） =================
OUTDIR = "/home/weiji/restart_exam/era5_data"
YEAR   = 2015                      # 要处理的年份（单年）
DATASET = "reanalysis-era5-pressure-levels"

VARIABLES = ["u_component_of_wind", "v_component_of_wind", "temperature"]
PLEV_1_100 = ['1','2','3','5','7','10','20','30','50','70','100']  # 1–100 hPa
PLEV_CHUNK_SIZE = 11              # 如遇 403，可调小到 4/3/2
GRID = "2.5/2.5"
FORMAT = "netcdf"
AREA = None                       # 例如仅 90S–90N: [90, -180, -90, 180]
COMP_LEVEL = 4

# 新增：按“多天窗口”下载（如 5 表示 5 天一组）
DAYS_PER_REQUEST = 5

# 只保留最终文件的策略
ONLY_KEEP_MONTHLY_FINAL = True    # 月度完成后，删除所有“窗口日均”和“小时块”中间件
ONLY_KEEP_YEARLY_FINAL  = True    # 年度完成后，删除所有“月度日均”，仅保留“年度日均”

# ================= 工具函数 =================
def days_in_month(y, m):
    if m == 12:
        return 31
    return (date(y if m < 12 else y + 1, m % 12 + 1, 1) - date(y, m, 1)).days

def chunks(lst, n):
    it = iter(lst)
    while True:
        block = list(itertools.islice(it, n))
        if not block:
            return
        yield block

def day_windows(y, m, k):
    nd = days_in_month(y, m)
    start = 1
    while start <= nd:
        end = min(start + k - 1, nd)
        yield [f"{d:02d}" for d in range(start, end + 1)], f"{start:02d}-{end:02d}"
        start = end + 1

def req_window(year, month, var, plev_block, day_list):
    req = {
        "product_type": "reanalysis",
        "variable": [var],
        "pressure_level": plev_block,
        "year": f"{year}",
        "month": [f"{month:02d}"],
        "day": day_list,
        "time": [f"{h:02d}:00" for h in range(24)],
        "grid": GRID,
        "format": FORMAT,
    }
    if AREA is not None:
        req["area"] = AREA
    return req

def retrieve_with_retry(c, dataset, req, target_path, max_retries=5):
    for i in range(max_retries):
        try:
            c.retrieve(dataset, req, target_path)
            return
        except Exception as e:
            if i == max_retries - 1:
                raise
            wait_s = 8 * (2 ** i)
            print(f"[Retry {i+1}/{max_retries-1}] {e}\nSleep {wait_s}s …")
            time.sleep(wait_s)

# 统一ERA5文件：时间坐标、去掉expver/number、排序去重
def _standardize_era5(ds: xr.Dataset) -> xr.Dataset:
    tname = "time" if "time" in ds.coords else ("valid_time" if "valid_time" in ds.coords else None)
    if tname is None:
        raise KeyError("No 'time' or 'valid_time' coord found in dataset.")
    if tname != "time":
        ds = ds.rename({tname: "time"})
    if "expver" in ds.dims:
        ds = ds.squeeze("expver", drop=True) if ds.sizes["expver"] == 1 else ds.sel(expver=ds["expver"].max()).drop_vars("expver", errors="ignore")
    if "number" in ds.dims:
        ds = ds.squeeze("number", drop=True) if ds.sizes["number"] == 1 else ds.isel(number=0).drop_vars("number", errors="ignore")
    ds = ds.sortby("time")
    try:
        idx = ds.get_index("time")
        if getattr(idx, "is_monotonic_increasing", True) is False or idx.has_duplicates:
            ds = ds.isel(time=~idx.duplicated())
    except Exception:
        pass
    return ds

# 辅助：生成路径
def hourly_chunk_path(year, month, dtag, var, plev_block):
    return os.path.join(OUTDIR, f"era5_hourly_{year}{month:02d}_d{dtag}_{var}_plev{'-'.join(plev_block)}_2p5deg.nc")

def window_daily_path(year, month, dtag):
    return os.path.join(OUTDIR, f"era5_dailymean_tuv_{year}{month:02d}_d{dtag}_1to100hPa_2p5deg.nc")

def monthly_daily_path(year, month):
    return os.path.join(OUTDIR, f"era5_dailymean_tuv_{year}{month:02d}_1to100hPa_2p5deg.nc")

def yearly_daily_path(year):
    return os.path.join(OUTDIR, f"era5_dailymean_tuv_{year}_1to100hPa_2p5deg.nc")

# ================= 主流程 =================
def main():
    os.makedirs(OUTDIR, exist_ok=True)
    c = cdsapi.Client()

    monthly_files = []

    for month in range(1, 13):
        print(f"\n================  {YEAR}-{month:02d}  ================")
        window_files = []

        for day_list, dtag in day_windows(YEAR, month, DAYS_PER_REQUEST):
            print(f"\n--- Window {YEAR}-{month:02d} days {dtag} ---")
            part_files = []

            for var in VARIABLES:
                for plev_block in chunks(PLEV_1_100, PLEV_CHUNK_SIZE):
                    hourly_nc = hourly_chunk_path(YEAR, month, dtag, var, plev_block)
                    if not os.path.exists(hourly_nc):
                        req = req_window(YEAR, month, var, plev_block, day_list)
                        print(f" -> Request {os.path.basename(hourly_nc)}")
                        retrieve_with_retry(c, DATASET, req, hourly_nc)
                    else:
                        print(f" -> Skip exists {os.path.basename(hourly_nc)}")
                    part_files.append(hourly_nc)

            # 合并窗口的小时块 -> 窗口逐日日均
            print(f" Merging {len(part_files)} hourly chunks for {YEAR}-{month:02d} days {dtag} …")
            ds_hourly = xr.open_mfdataset(part_files, combine="by_coords", preprocess=_standardize_era5)
            win_daily_nc = window_daily_path(YEAR, month, dtag)
            ds_daily = ds_hourly.resample(time="1D").mean(keep_attrs=True)
            enc = {v: {"zlib": True, "complevel": COMP_LEVEL} for v in ds_daily.data_vars}
            ds_daily.to_netcdf(win_daily_nc, encoding=enc)
            ds_hourly.close(); del ds_hourly, ds_daily
            window_files.append(win_daily_nc)

            # 清理窗口的小时中间件
            for f in part_files:
                try: os.remove(f)
                except Exception: pass

        # 合并所有窗口 -> 月度日均
        print(f"\n=== Merge all window daily means -> monthly {YEAR}-{month:02d} ===")
        ds_month = xr.open_mfdataset(window_files, combine="by_coords", preprocess=_standardize_era5)
        mon_nc = monthly_daily_path(YEAR, month)
        enc = {v: {"zlib": True, "complevel": COMP_LEVEL} for v in ds_month.data_vars}
        ds_month.to_netcdf(mon_nc, encoding=enc)
        ds_month.close(); del ds_month
        monthly_files.append(mon_nc)

        # 清理窗口级日均文件
        if ONLY_KEEP_MONTHLY_FINAL:
            for f in window_files:
                try: os.remove(f)
                except Exception: pass

    # 合并 12 个月 -> 年度日均
    print(f"\n================  Merge all months -> yearly {YEAR}  ================")
    ds_year = xr.open_mfdataset(monthly_files, combine="by_coords", preprocess=_standardize_era5)
    yr_nc = yearly_daily_path(YEAR)
    enc = {v: {"zlib": True, "complevel": COMP_LEVEL} for v in ds_year.data_vars}
    ds_year.to_netcdf(yr_nc, encoding=enc)
    ds_year.close(); del ds_year

    # 清理月度文件（若只保留年度最终）
    if ONLY_KEEP_YEARLY_FINAL:
        for f in monthly_files:
            try: os.remove(f)
            except Exception: pass

    print(f"\n✅ Year done: {yr_nc}")
    if ONLY_KEEP_YEARLY_FINAL:
        print("（已清理所有中间与月度文件，仅保留年度最终文件）")

if __name__ == "__main__":
    main()
